## Spam Email detection by TF-IDF

Classification of Mail Messages as Spam/Not Spam using logistic regression ML Model & TF-IDF NLP Technique

| **Column Name** | **Description**                                                      | **Example**                                                             |
| --------------- | -------------------------------------------------------------------- | ----------------------------------------------------------------------- |
| `Category`      | The label showing whether the mail is **Spam** or **Ham (Not Spam)** | `Spam`, `Ham`                                                           |
| `Message`       | The actual **email content or message text**                         | `"Congratulations! You’ve won a $1000 gift card. Click here to claim!"` |


In [21]:
## importing dependencies
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

**Data Collection & Preproccessing**

In [22]:
## Loading the Dataset 
df = pd.read_csv("mail_data.csv")

In [23]:
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [24]:
df.shape ## 5572 records

(11572, 2)

In [25]:
## Checking for null values
df.isnull().sum()

Category    0
Message     0
dtype: int64

In [26]:
## Checking for duplicate values
df.duplicated().sum() ## 415 duplicate values

np.int64(4384)

In [27]:
## Dropping duplicate values
df.drop_duplicates(inplace=True)

In [28]:
## Text cleaning steps

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # remove URLs
    text = re.sub(r'\W', ' ', text)  # remove punctuation
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

df['Message'] = df['Message'].apply(clean_text)


In [29]:
df.shape

(7188, 2)

**Label Encoding**

In [30]:
## Label Encoding category column - 0 ham & 1 for Spam
df["Category"].value_counts()
df["Category"] = df["Category"].apply(lambda x:1 if x=="ham" else 0)
df["Category"] = df["Category"].astype("int")

In [31]:
df.head()

,Category,Message
0,1,go until jurong point crazy available only in ...
1,1,ok lar joking wif u oni
2,0,free entry in a wkly comp to win fa cup final ...
3,1,u dun say so early hor u c already then say
4,1,nah i don t think he goes to usf he lives arou...


In [32]:
df[df["Category"]==0]

,Category,Message
2,0,free entry in a wkly comp to win fa cup final ...
5,0,freemsg hey there darling it s been week s now...
8,0,winner as a valued network customer you have b...
9,0,had your mobile months or more u r entitled to...
11,0,six chances to win cash from to pounds txt csh...
...,...,...
9556,0,your parcel is on hold due to unpaid fee of
9565,0,delivery attempt failed pay to reschedule
9567,0,delivery attempt failed pay to reschedule
9568,0,delivery attempt failed pay to reschedule


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7188 entries, 0 to 10590
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  7188 non-null   int64 
 1   Message   7188 non-null   object
dtypes: int64(1), object(1)
memory usage: 168.5+ KB


**Test-Train Split**

In [34]:
## Defining X & y
X = df["Message"]
y = df["Category"]

In [35]:
## Splitting the Data
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=2)

In [36]:
X.shape ## checking whether X is in Dataframe format

(7188,)

**Training the Model**

In [37]:
# -------- PIPELINE --------
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1,2)
    )),
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        use_label_encoder=False
    ))
])

In [38]:
pipeline.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('xgb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


**Evaluating the Trained Model**

In [39]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# -------- PREDICTIONS --------
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:,1]

# -------- METRICS --------
print("\n📊 MODEL EVALUATION\n")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))



📊 MODEL EVALUATION

Accuracy: 0.9756606397774688
Precision: 0.9657754010695188
Recall: 0.9966887417218543
F1 Score: 0.9809885931558935
ROC-AUC: 0.9945341416455045

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.94      0.97       532
           1       0.97      1.00      0.98       906

    accuracy                           0.98      1438
   macro avg       0.98      0.97      0.97      1438
weighted avg       0.98      0.98      0.98      1438


Confusion Matrix:

[[500  32]
 [  3 903]]


## Thank you